# CM-204 · Pipeline de Metric Learning — Layers 0-3 · **Kaggle (execução única, GPU T4)**

Notebook unificado: as Layers 0-1 (antes rodadas no Colab) e as Layers 2-3 (antes rodadas
no Kaggle a partir de um dataset com cache restaurado) agora rodam **numa única sessão**,
do início ao fim, sem reimportar dependências nem repetir setup. **O código mora no
repositório git**; este notebook apenas o executa, célula a célula, de cima para baixo.

**Fluxo:** setup → download do CUB-200-2011 → extração de embeddings (Layer 0 cache) →
baseline congelado (Layer 0) → ablação de normalização → cabeça treinada (Layer 1) →
diagnósticos da Layer 1 → ablação de mineração (Layer 2) → sanidade + fine-tuning LoRA
(Layer 3) → tabela final consolidada (Layers 0-3) → figuras → resgate dos artefatos.

> **Antes de começar** (painel direito): **Accelerator → GPU T4** (x2 ou simples — **não
> use P100**: a versão do PyTorch pré-instalada no Kaggle é incompatível com P100) e
> **Internet → On** (necessário para o clone do repo, o `pip install` e o download do CUB).
> Não é preciso montar nenhum dataset extra — o CUB é baixado da internet e o cache das
> Layers 0-1 é gerado nesta própria sessão, então a etapa de "restaurar artefatos de uma
> sessão anterior" do notebook do Kaggle original foi eliminada.
>
> **Ao final, rode a última seção** para copiar `results/`, `configs/default.yaml` e o
> `.zip` consolidado para `/kaggle/working` — é o que fica disponível na aba **Output**
> após o commit (o disco de sessão do Kaggle, assim como o do Colab, é efêmero).
>
> **Tempo/GPU:** a Layer 3 (fine-tuning LoRA) é a etapa cara — cada época faz um forward
> completo do ViT sobre as imagens de treino. As Layers 0, 1 e 2 rodam sobre o cache
> (CPU/GPU leve) em minutos. Para rodar tudo de uma vez em background: **Save Version →
> Save & Run All (Commit) → Save**.


## 1 · Setup do ambiente

Feito **uma única vez** para o notebook inteiro — nada aqui se repete mais adiante.

In [ ]:
# GPU da sessão Kaggle. Selecione T4 no painel direito (Accelerator) — a Layer 3
# depende dela, e P100 é incompatível com a versão do PyTorch pré-instalada no Kaggle.
!nvidia-smi

In [ ]:
# Código do repositório (Internet deve estar On no painel direito). Não editar
# código dentro do notebook — o código mora no repositório.
!git clone https://github.com/RuiACNunes/CM204_RuiACN.git
%cd CM204_RuiACN

In [ ]:
# Dependências. O Kaggle já traz torch com CUDA — NÃO reinstalar torch, ou o
# ambiente quebra.
!pip install -q open_clip_torch faiss-cpu pyyaml pandas tabulate tqdm Pillow

## 2 · Dataset CUB-200-2011

Baixa e extrai o `CUB_200_2011.tgz` (~1.1 GB) oficial, com **três fontes em fallback**
(Caltech, mirror GitHub, Google Drive) — os datasets de CUB prontos no Kaggle não servem
(220 classes ou sem os metadados que o loader exige). Idempotente: se a pasta já existir,
o download é pulado. Valida a estrutura oficial (`images.txt`, `image_class_labels.txt`,
`classes.txt`, `images/`) e o MD5 ao final.

> Leva alguns minutos (download + extração). É o preço de ter o dataset **correto e
> comparável** entre todas as Layers — a mesma cópia gera o cache de embeddings (Layers
> 0-2) e alimenta o fine-tuning (Layer 3).

In [ ]:
# Download do CUB-200-2011 oficial, com fallback de fontes. Requer Internet -> On.
import os, subprocess, hashlib, glob

CUB_ROOT = 'data/raw/CUB_200_2011'          # = paths.cub_root no config
TGZ      = 'data/raw/CUB_200_2011.tgz'
MD5_OK   = '97eceeb196236b17998738112f37df78'   # md5 oficial do .tgz
os.makedirs('data/raw', exist_ok=True)

def _have_cub():
    return all(os.path.exists(os.path.join(CUB_ROOT, n)) for n in
               ['images.txt', 'image_class_labels.txt', 'classes.txt']) \
           and os.path.isdir(os.path.join(CUB_ROOT, 'images'))

if _have_cub():
    print("CUB ja presente - download pulado.")
else:
    SOURCES = [
        ("Caltech (oficial)",
         "https://www.vision.caltech.edu/datasets/CUB-200-2011/CUB_200_2011.tgz"),
        ("Caltech (visipedia-data)",
         "http://www.vision.caltech.edu/visipedia-data/CUB-200-2011/CUB_200_2011.tgz"),
        ("GitHub mirror (vignagajan)",
         "https://media.githubusercontent.com/media/vignagajan/CUB-200-2011/main/CUB_200_2011.tgz"),
    ]
    ok = False
    for nome, url in SOURCES:
        print(f"-> tentando {nome}")
        r = subprocess.run(["wget", "-q", "--show-progress", "--timeout=60",
                            "--tries=2", "-O", TGZ, url])
        if r.returncode == 0 and os.path.exists(TGZ) and os.path.getsize(TGZ) > 1_000_000_000:
            print(f"   baixado ({os.path.getsize(TGZ)/1e9:.2f} GB)"); ok = True; break
        print("   falhou - proxima fonte")
    if not ok:
        print("-> tentando Google Drive (gdown)...")
        subprocess.run(["pip", "install", "-q", "gdown"])
        import gdown
        gdown.download(id="1hbzc_P1FuxMkcabkgn9ZKinBwW683j45", output=TGZ, quiet=False)
        ok = os.path.exists(TGZ) and os.path.getsize(TGZ) > 1_000_000_000
    assert ok, ("Todas as fontes falharam. Verifique Internet -> On, ou suba o "
                "CUB_200_2011.tgz como dataset.")

    print("verificando MD5...")
    h = hashlib.md5()
    with open(TGZ, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    print("   md5:", h.hexdigest(), "OK" if h.hexdigest() == MD5_OK else "(difere, prossigo)")

    print("extraindo...")
    subprocess.run(["tar", "-xzf", TGZ, "-C", "data/raw"], check=True)

assert _have_cub(), f"estrutura oficial nao encontrada em {CUB_ROOT} apos extracao"
n_jpg = len(glob.glob(f'{CUB_ROOT}/images/**/*.jpg', recursive=True))
especies = sorted(os.path.basename(d) for d in glob.glob(f'{CUB_ROOT}/images/*'))
print(f"\nCUB OK - imagens .jpg: {n_jpg}  [esperado: 11788] - classes: {len(especies)}")
print("primeiras classes:", especies[:3])

In [ ]:
# Verificar o download ANTES de gastar GPU.
!ls -lh data/raw/CUB_200_2011.tgz          # ~1.1 GB
!wc -l data/raw/CUB_200_2011/images.txt    # deve dar 11788
!wc -l data/raw/CUB_200_2011/classes.txt   # deve dar 200

## 3 · Extração de embeddings (Layer 0 — cache)

Passa as 11.788 imagens pelo CLIP ViT-B/16 congelado **uma vez** e salva um tensor
`[11788, 512]` **raw (não normalizado)** em `data/embeddings/clip_vitb16.npz`. A
normalização é aplicada downstream, por consumidor. A partir daqui, as Layers 0-2 não
precisam mais de GPU — só a Layer 3 volta a usá-la, para o fine-tuning.

O `device` é resolvido por `"auto"` no config, e o backbone corrige o nome do modelo
para `ViT-B-16-quickgelu` (pesos OpenAI). Ambos já estão no repositório — sem remendos.

In [ ]:
# Extrair (minutos com T4). O aviso 'QuickGELU ... usando ViT-B-16-quickgelu' é
# esperado — é a correção da ativação disparando corretamente.
!python scripts/00_cache_embeddings.py --config configs/default.yaml

In [ ]:
# Sanity check do cache: shape, intervalo de classes, cache raw e split disjunto.
import numpy as np

d = np.load('data/embeddings/clip_vitb16.npz')
e, c = d['embeddings'], d['class_id']

print(f"shape:          {e.shape}  ({e.dtype})            [esperado: (11788, 512) float32]")
print(f"class_id range: {c.min()} -> {c.max()}                       [esperado: 1 -> 200]")
print(f"classes unicas: {len(np.unique(c))}                             [esperado: 200]")
print(f"norma media:    {np.linalg.norm(e, axis=1).mean():.3f}                    [>>1 => raw, correto]")
print(f"NaN / Inf:      {np.isnan(e).any()} / {np.isinf(e).any()}                 [esperado: False / False]")

# Disjuncao do split metric-learning: nenhuma classe nos dois lados.
train_mask, test_mask = c <= 100, c >= 101
assert not (set(c[train_mask]) & set(c[test_mask])), "SPLIT CONTAMINADO"
print(f"\nsplit metric-learning: {train_mask.sum()} treino / {test_mask.sum()} teste - disjunto OK")

## 4 · Layer 0 — Baseline congelado + kNN

O piso de referência, sem treino. Avaliação leave-one-out sobre o split de teste (classes 101–200). Grava a linha `frozen_knn` em `results/experiments.csv`.

In [ ]:
!python scripts/01_baseline_knn.py --config configs/default.yaml

In [ ]:
# Leitura critica do baseline - nao siga sem interpretar.
import pandas as pd

df = pd.read_csv('results/experiments.csv')
base = df[df.regime == 'frozen_knn'].iloc[-1]
r1 = base['R@1']

print(base[['regime', 'R@1', 'R@2', 'R@4', 'R@8', 'mAP@R']].to_string())

if r1 > 0.95:
    print("\n[!] R@1 ~ 100% - leave-one-out falhou: a query esta se encontrando na gallery.")
elif r1 < 0.05:
    print("\n[!] R@1 ~ aleatorio - bug no split ou desalinhamento embeddings/rotulos.")
elif 0.40 <= r1 <= 0.75:
    print(f"\n[OK] R@1 = {r1:.1%} - faixa plausivel para CLIP ViT-B congelado no CUB. Siga.")
else:
    print(f"\n[?] R@1 = {r1:.1%} - fora da faixa tipica. Investigar antes de treinar.")

## 5 · Ablação de normalização (diagnóstico)

Roda o baseline uma segunda vez **sem** L2-normalização, gerando uma segunda linha
`frozen_knn` (com `l2_normalize=False`). Mostra o efeito de medir distância euclidiana
sobre vetores de norma variável vs. cosseno. Como o cache é raw, não é preciso recachear.

> **Atenção:** a partir daqui existem DUAS linhas `frozen_knn` no CSV. As células
> seguintes usam explicitamente `l2_normalize == True` como baseline canônico — não
> use `.iloc[-1]` cru, que pegaria o não-normalizado.

In [ ]:
# sed e remendo de sessao; o resultado (baseline nao-normalizado) e o que interessa.
!sed -i 's/baseline_l2_normalize: true/baseline_l2_normalize: false/' configs/default.yaml
!python scripts/01_baseline_knn.py --config configs/default.yaml
!sed -i 's/baseline_l2_normalize: false/baseline_l2_normalize: true/' configs/default.yaml
!grep -n "baseline_l2_normalize" configs/default.yaml   # confirmar que voltou para true

In [ ]:
# As duas linhas de baseline, lado a lado.
import pandas as pd
final = pd.read_csv('results/experiments.csv')
print(final[final.regime == 'frozen_knn'][['timestamp', 'l2_normalize', 'R@1', 'mAP@R']].to_string())

## 6 · Layer 1 — Cabeça de projeção treinada

Treina um MLP (512 → 128, saída L2-normalizada) sobre os embeddings congelados, com triplet loss. Uma head por estratégia de mineração (`batch_hard`, `batch_all`), seed resetada a cada uma. Persiste curvas de convergência e a linha final por regime.

In [ ]:
!python scripts/02_train_head.py --config configs/default.yaml

In [ ]:
# Diagnostico de colapso - o modo de falha classico do batch_hard.
import glob, os
import pandas as pd

PREFIX = 'training_head_triplet_'
for path in sorted(glob.glob(f'results/{PREFIX}*.csv')):
    t = pd.read_csv(path)
    strat = os.path.basename(path)[len(PREFIX):-len('.csv')]
    print(f"\n-- {strat} --")
    print(t.tail(3).to_string(index=False))
    if 'R@1' in t and t['R@1'].iloc[-1] < t['R@1'].iloc[0]:
        print("  [!] metrica piorou ao longo do treino - possivel colapso/sobreajuste.")

## 7 · Diagnósticos da Layer 1 (dois testes baratos)

A Layer 1 degrada o R@1 vs. o baseline. Estes dois testes separam as explicações
concorrentes, e decidem o que se escreve na discussão do relatório:

- **Teste A — head aleatória (época 0):** isola o efeito do gargalo 512→128 do efeito
  do treino. Se a head não treinada já ~iguala o baseline, o treino é que degrada;
  se ela já cai muito, o gargalo dimensional é o culpado.
- **Teste B — margem maior:** a loss ~0 sugere que margem 0.2 é frouxa demais sobre
  vetores unitários. Se `margin=0.5` recupera o R@1, era hiperparâmetro, não achado.

Ambos rodam em segundos/minutos sobre o cache. O Teste B grava num CSV **separado**
para não poluir a tabela oficial.

In [ ]:
# -- Diagnosticos da Layer 1 --
import copy
import numpy as np
import pandas as pd
import torch
import yaml

from src.models.projection import ProjectionHead
from src.eval.retrieval import leave_one_out_ranking
from src.eval.metrics import evaluate_retrieval
from src.models.backbone import _resolve_device
from src.utils.seed import set_seed
from src.training.head_trainer import train_head

# Config e cache (autocontido - nao depende de variaveis de celulas anteriores)
config = yaml.safe_load(open('configs/default.yaml'))

d = np.load('data/embeddings/clip_vitb16.npz')
emb = torch.tensor(d['embeddings'], dtype=torch.float32)
cid = torch.tensor(d['class_id'].astype(np.int64))
train_mask = (cid <= 100).numpy()
test_mask  = (cid >= 101).numpy()

device        = _resolve_device(config['training']['device'])
retrieval_cfg = config['retrieval']
head_cfg      = config['head']
test_embeddings = emb[test_mask].numpy()
test_labels     = cid[test_mask].numpy()

# Baseline canonico (L2) para referencia
final = pd.read_csv(config['logging']['experiments_csv'])
b = final.query("regime == 'frozen_knn' and l2_normalize == True").iloc[-1]
print(f"referencia frozen (L2):  R@1 = {b['R@1']:.3f}   mAP@R = {b['mAP@R']:.3f}\n")

# ===== TESTE A - head aleatoria (epoca 0) =====
set_seed(config['seed'])
head0 = ProjectionHead(in_dim=head_cfg['in_dim'], hidden_dim=head_cfg['hidden_dim'],
                       out_dim=head_cfg['out_dim']).to(device).eval()
with torch.no_grad():
    proj0 = head0(torch.tensor(test_embeddings, dtype=torch.float32, device=device)).cpu().numpy()
rank0 = leave_one_out_ranking(proj0, metric=retrieval_cfg['metric'])
m0 = evaluate_retrieval(rank0, test_labels, retrieval_cfg['recall_ks'])
print("TESTE A - head aleatoria (epoca 0, sem treino):")
print(f"   R@1 = {m0['R@1']:.3f}   mAP@R = {m0['mAP@R']:.3f}")
if m0['R@1'] >= b['R@1'] - 0.03:
    print("   -> projecao aleatoria ja ~iguala o baseline. O TREINO degrada, nao o gargalo.\n")
elif m0['R@1'] <= 0.55:
    print("   -> gargalo 512->128 sozinho ja custa muito. O treino melhora sobre isto, depois piora.\n")
else:
    print("   -> intermediario; leia junto com as curvas de convergencia.\n")

# ===== TESTE B - margem maior =====
cfg_b = copy.deepcopy(config)
cfg_b['training']['margin'] = 0.5
cfg_b['logging'] = dict(cfg_b['logging'])
cfg_b['logging']['experiments_csv']  = 'results/_diag_margin05.csv'
cfg_b['logging']['experiments_json'] = 'results/_diag_margin05.json'

payload = {'embeddings': emb, 'class_id': cid}
print("TESTE B - batch_hard com margin=0.5:")
m_b = train_head(config=cfg_b, payload=payload, train_mask=train_mask, test_mask=test_mask,
                 strategy='batch_hard', training_log_dir='results')
print(f"\n   margin=0.5 -> R@1 = {m_b['R@1']:.3f}   mAP@R = {m_b['mAP@R']:.3f}")
print(f"   (baseline L2 = {b['R@1']:.3f};  batch_hard margin=0.2 foi ~0.575)")
if m_b['R@1'] > 0.60:
    print("   -> margem maior recupera R@1: era HIPERPARAMETRO, nao achado de generalizacao.")
else:
    print("   -> margem maior nao salva: reforca sobreajuste/gargalo como explicacao real.")

## 8 · Layer 2 — Ablação de mineração

Roda as **quatro** estratégias de mineração sobre o cache da Layer 0. O backbone não
entra no grafo — tudo em CPU sobre embeddings fixos, como na Layer 1.

| estratégia | ideia | papel na narrativa |
|---|---|---|
| `batch_all` | todos os triplos válidos | piso: maioria é trivial (grad. nulo) |
| `batch_hard` | negativo mais difícil por âncora | teto agressivo (risco de colapso) |
| `semi_hard` | negativo mais distante que o positivo, dentro da margem (FaceNet) | meio-termo canônico |
| `distance_weighted` | amostragem ponderada pela distância (Wu 2017) | corrige o viés de negativos triviais |

`batch_hard` e `batch_all` já rodaram na Layer 1; são incluídos aqui para a tabela ficar
completa e comparável. `margin` só afeta `semi_hard`.

In [ ]:
# As quatro estrategias, sequencial (~4x o tempo da Layer 1, tudo em CPU).
# Grava 4 linhas em results/experiments.csv + 4 curvas training_head_triplet_<strategy>.csv
!python scripts/04_mining_ablation.py --config configs/default.yaml

In [ ]:
# Leitura da ablacao: as 4 estrategias contra o baseline L2 canonico.
import pandas as pd

df = pd.read_csv('results/experiments.csv')
b = df.query("regime == 'frozen_knn' and l2_normalize == True").iloc[-1]
print(f"baseline frozen (L2):  R@1 = {b['R@1']:.3f}   mAP@R = {b['mAP@R']:.3f}\n")

abl = df[df.regime.str.startswith('head_triplet_')].copy()
abl['strategy'] = abl['mining_strategy']
view = abl.groupby('strategy', as_index=False).last()[
    ['strategy', 'margin', 'R@1', 'R@2', 'R@4', 'R@8', 'mAP@R']
].sort_values('R@1', ascending=False)
print(view.to_string(index=False))

best = view.iloc[0]
print(f"\nmelhor estrategia por R@1: {best['strategy']} ({best['R@1']:.3f})")
if best['R@1'] < b['R@1']:
    print("[!] nenhuma estrategia supera o frozen L2 - o gargalo 512->128 domina.")
    print("    Este e o achado que motiva a Layer 3: adaptar o backbone, nao so projetar.")
else:
    print("[OK] mineracao recupera terreno sobre o baseline - reportar a vencedora.")

In [ ]:
# Curvas de convergencia das 4 estrategias, R@1 e mAP@R lado a lado.
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

PREFIX = 'training_head_triplet_'
curves = sorted(glob.glob(f'results/{PREFIX}*.csv'))
final = pd.read_csv('results/experiments.csv')
b = final.query("regime == 'frozen_knn' and l2_normalize == True").iloc[-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for path in curves:
    t = pd.read_csv(path)
    strat = os.path.basename(path)[len(PREFIX):-len('.csv')]
    axes[0].plot(t['epoch'], t['R@1'],   marker='o', ms=3, label=strat)
    axes[1].plot(t['epoch'], t['mAP@R'], marker='o', ms=3, label=strat)

axes[0].axhline(b['R@1'],   ls='--', c='gray', label='frozen (Layer 0)')
axes[1].axhline(b['mAP@R'], ls='--', c='gray', label='frozen (Layer 0)')
axes[0].set_ylabel('Recall@1'); axes[1].set_ylabel('mAP@R')
for ax in axes:
    ax.set_xlabel('epoca'); ax.grid(alpha=.3); ax.legend()

plt.tight_layout()
plt.savefig('results/ablacao_layer2.png', dpi=200)
plt.show()

## 9 · Layer 3 — Sanidade da injeção LoRA (antes de gastar GPU)

Antes do treino longo, dois checks baratos que pegam os erros clássicos de PEFT:

1. **Inicialização identidade:** `B=0` ⇒ o delta LoRA é zero no passo 0, então o modelo
   adaptado deve produzir **exatamente** o mesmo embedding que o CLIP original. Se
   divergir, a injeção está errada.
2. **Contagem de treináveis:** só os adaptadores devem ter `requires_grad=True`. A fração
   treinável tem que ficar na casa de **~0.1%** (PEFT); qualquer valor alto significa que
   o backbone não foi congelado.

Ambos rodam num único forward — segundos. As imagens do disco (baixadas na seção 2) são
usadas a partir daqui — é a única parte do notebook que precisa delas, além da GPU.

In [ ]:
# -- Sanidade da injecao LoRA (1 batch, sem treino) --
# Requer src/models/lora.py corrigido: injeta em mlp.c_proj (nao em attn.out_proj, que o
# nn.MultiheadAttention le como .weight/.bias e nunca chama como modulo) e congela o modelo
# inteiro antes de injetar (os LoRALinear reabrem so A/B).
import torch, yaml
from src.models.backbone import ClipBackbone
from src.models.lora import LoRAConfig, inject_lora, count_parameters

config = yaml.safe_load(open('configs/default.yaml'))
bcfg, lcfg = config['backbone'], config['lora']

backbone = ClipBackbone(
    model_name=bcfg['model_name'],
    pretrained=bcfg['pretrained'],
    device=bcfg['device'],
)
device = backbone.device
x = torch.randn(4, 3, 224, 224, device=device)

# Embedding do CLIP original, antes da injecao (encode() ja e no_grad + raw).
emb_orig = backbone.encode(x)

# Injetar LoRA - B=0 => delta=0 => embedding identico no passo 0.
inject_lora(backbone.model, LoRAConfig(
    rank=lcfg['rank'], alpha=lcfg['alpha'], dropout=lcfg['dropout'],
))
backbone.model.to(device)
emb_lora = backbone.encode(x)

max_diff = (emb_orig - emb_lora).abs().max().item()
print(f"1. init identidade - max |delta embedding| = {max_diff:.2e}   [esperado: ~0]")
assert max_diff < 1e-4, "LoRA nao parte do CLIP original - B deveria ser inicializado com zeros"

# Contagem de treinaveis no modelo todo (fiel: inject_lora congela o modelo inteiro).
trainable, total = count_parameters(backbone.model)
print(f"2. treinaveis = {trainable:,} / {total:,}  ({100*trainable/total:.4f}%)   [esperado: ~0.12%]")
assert trainable < 0.01 * total, "backbone nao foi congelado - fracao treinavel alta demais"
print("\nsanidade OK - pode treinar.")

In [ ]:
# Ajusta o config para esta sessao: 40 epocas e num_workers=4 (Kaggle T4 tem 4 vCPUs).
# Edita o YAML in-place para que os dois runs (r=4 e r=8) usem os mesmos hiperparametros.
import yaml

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['lora']['epochs'] = 40
cfg['lora']['num_workers'] = 4

with open('configs/default.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print(f"lora.epochs = {cfg['lora']['epochs']}   lora.num_workers = {cfg['lora']['num_workers']}")
print(f"lora.eval_every = {cfg['lora'].get('eval_every')}   (avaliacao a cada N epocas)")

## 10 · Layer 3 — Fine-tuning com LoRA (GPU)

Agora o backbone entra no grafo. Cada passo passa **imagens** pelo CLIP adaptado, projeta
pela head e retropropaga pelos adaptadores LoRA (o resto do backbone continua congelado).

**Adaptadores em `mlp.c_proj`.** O open_clip funde Q/K/V num único `in_proj_weight` (não é
`nn.Linear`), e o `out_proj` da atenção não pode receber o adaptador: o
`nn.MultiheadAttention` do PyTorch lê `out_proj.weight`/`.bias` diretamente na função nativa
em vez de chamar o módulo, então um wrapper ali nunca executaria. Injeta-se LoRA na projeção
de saída do bloco feed-forward (`mlp.c_proj`), que é um `nn.Linear` de verdade chamado como
módulo. Adaptar as camadas MLP é prática PEFT padrão; para rank=4 são ~184k parâmetros
treináveis (0.12% do backbone).

**Dois grupos de parâmetros:** LoRA a `lr=1e-4`, head a `lr=1e-3` (escalas de gradiente
diferentes). `backbone.model.eval()` é mantido no treino todo (congela BN/Dropout dos
layers base); LoRA e head não têm BN, então o modo eval não os afeta.

> **Tempo/GPU:** cada época faz um forward completo do ViT sobre as imagens de treino —
> ordens de grandeza acima das Layers 0-2. Os dois runs (`rank=4` e `rank=8`) rodam em
> sequência nesta seção. Se o tempo de sessão apertar, comente a célula do segundo run
> (r=8) e trate o sweep de rank como trabalho futuro no relatório — um único `r=4` já
> sustenta a tese central (LoRA supera o frozen com <0.2% dos parâmetros).

In [ ]:
# Run 1 de 2: LoRA rank=4 (default do config).
# Grava linha lora_triplet_batch_hard_r4 + curva training_lora_triplet_batch_hard_r4.csv
!python scripts/05_train_lora.py --config configs/default.yaml --rank 4

In [ ]:
# Diagnostico rapido do run principal antes de gastar GPU no sweep.
import pandas as pd

curve = 'results/training_lora_triplet_batch_hard_r4.csv'
t = pd.read_csv(curve)
print(t.to_string(index=False))

df = pd.read_csv('results/experiments.csv')
b = df.query("regime == 'frozen_knn' and l2_normalize == True").iloc[-1]
last = t.iloc[-1]
d_r1 = (last['R@1'] - b['R@1']) / b['R@1']
print(f"\nfinal LoRA r=4:  R@1 = {last['R@1']:.3f}   mAP@R = {last['mAP@R']:.3f}")
print(f"vs. frozen L2 ({b['R@1']:.3f}):  DeltaR@1 = {d_r1:+.1%}")
if last['R@1'] > b['R@1']:
    print("[OK] LoRA supera o frozen - adaptar o backbone paga. E o achado central do relatorio.")
else:
    print("[?] LoRA ainda nao supera o frozen - checar epocas/lr antes do sweep, ou reportar honestamente.")

In [ ]:
# Segundo run: LoRA rank=8 (o r=4 rodou acima). Dois pontos = ablacao r=4 vs r=8.
# Grava linha lora_triplet_batch_hard_r8 + curva training_lora_triplet_batch_hard_r8.csv
!python scripts/05_train_lora.py --config configs/default.yaml --rank 8

# (opcional, bonus se sobrar cota) rank maior ou outra estrategia:
# !python scripts/05_train_lora.py --config configs/default.yaml --rank 16
# !python scripts/05_train_lora.py --config configs/default.yaml --rank 8 --strategy semi_hard

In [ ]:
# Curvas do sweep de rank: R@1 e mAP@R por epoca, com a linha do baseline frozen.
import glob, os, re
import pandas as pd
import matplotlib.pyplot as plt

PREFIX = 'training_lora_triplet_'
curves = sorted(glob.glob(f'results/{PREFIX}*.csv'))
if not curves:
    raise FileNotFoundError('Nenhuma curva LoRA - o script 05 rodou?')

final = pd.read_csv('results/experiments.csv')
b = final.query("regime == 'frozen_knn' and l2_normalize == True").iloc[-1]

def label_from(path):
    name = os.path.basename(path)[len(PREFIX):-len('.csv')]  # ex.: batch_hard_r4
    m = re.search(r'_r(\d+)$', name)
    return f"LoRA r={m.group(1)}" if m else f"LoRA {name}"

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for path in curves:
    t = pd.read_csv(path)
    lab = label_from(path)
    axes[0].plot(t['epoch'], t['R@1'],   marker='o', ms=3, label=lab)
    axes[1].plot(t['epoch'], t['mAP@R'], marker='o', ms=3, label=lab)

axes[0].axhline(b['R@1'],   ls='--', c='gray', label='frozen (Layer 0)')
axes[1].axhline(b['mAP@R'], ls='--', c='gray', label='frozen (Layer 0)')
axes[0].set_ylabel('Recall@1'); axes[1].set_ylabel('mAP@R')
for ax in axes:
    ax.set_xlabel('epoca'); ax.grid(alpha=.3); ax.legend()

plt.tight_layout()
plt.savefig('results/convergencia_layer3.png', dpi=200)
plt.show()

## 11 · Tabela final consolidada (Layers 0–3)

Todos os regimes numa tabela, ordenados por R@1, com deltas relativos ao baseline L2-normalizado. É a tabela-mãe do relatório: mostra o arco frozen → head → mineração → LoRA.

In [ ]:
import pandas as pd

df = pd.read_csv('results/experiments.csv')
b = df.query("regime == 'frozen_knn' and l2_normalize == True").iloc[-1]

# Uma linha por regime (ultima ocorrencia), baseline nao-normalizado fica de fora da tabela oficial.
mask = ~((df.regime == 'frozen_knn') & (df.l2_normalize == False))
tab = (df[mask]
       .groupby('regime', as_index=False).last()
       [['regime', 'R@1', 'R@2', 'R@4', 'R@8', 'mAP@R']]
       .sort_values('R@1', ascending=False))

tab['DeltaR@1']   = ((tab['R@1']   - b['R@1'])   / b['R@1']).map(lambda x: f"{x:+.1%}")
tab['DeltamAP@R'] = ((tab['mAP@R'] - b['mAP@R']) / b['mAP@R']).map(lambda x: f"{x:+.1%}")

print(tab.to_markdown(index=False))
tab.to_csv('results/summary_final.csv', index=False)
print("\n-> results/summary_final.csv gravado (para o relatorio).")

In [ ]:
# Tabela em markdown "crua", pronta para o relatorio (sem deltas, todas as ocorrencias).
import pandas as pd
df = pd.read_csv('results/experiments.csv')
print(df[['regime', 'R@1', 'R@2', 'R@4', 'R@8', 'mAP@R']].to_markdown(index=False))

## 12 · Resgatar artefatos (Kaggle)

O disco de sessão do Kaggle é efêmero, assim como o do Colab. Copie tudo para
`/kaggle/working/` — é o que fica anexado à versão em **Save & Run All (Commit)** e
disponível na aba **Output**. Não se usa `files.download` (isso é específico do Colab).

In [ ]:
# Copia results/ + config para /kaggle/working (raiz persistida pelo commit) e monta
# um unico zip, conveniente para baixar da aba Output.
import shutil, os

os.makedirs('/kaggle/working/results', exist_ok=True)
for f in os.listdir('results'):
    shutil.copy(f'results/{f}', f'/kaggle/working/results/{f}')
shutil.copy('configs/default.yaml', '/kaggle/working/default.yaml')

os.makedirs('/kaggle/working/embeddings', exist_ok=True)
shutil.copy('data/embeddings/clip_vitb16.npz', '/kaggle/working/embeddings/clip_vitb16.npz')

shutil.make_archive('/kaggle/working/artefatos', 'zip', 'results')
print("salvo em /kaggle/working - disponivel na aba Output apos o commit:")
print(sorted(os.listdir('/kaggle/working')))

---
### Rodar em background + estratégia de tempo

1. **Accelerator → GPU T4**, **Internet → On** (painel direito). Não é preciso adicionar
   nenhum dataset via **Add Input** — o CUB é baixado da internet e o cache das Layers
   0-1 é gerado nesta mesma sessão.
2. **Save Version → Save & Run All (Commit) → Save.** Feche a aba: roda em background e
   salva em `/kaggle/working`.
3. **Enquanto roda, comece a redação** com as seções de Introdução e Background do
   relatório, que não dependem de números. Preencha os resultados (Seções V-VI) quando o
   commit terminar.
4. **Se o tempo de sessão apertar:** comente a célula do segundo run LoRA (`rank=8`) na
   Seção 10 — um único `r=4` já sustenta a tese central do trabalho (LoRA supera o
   frozen com <0.2% dos parâmetros treináveis), e o sweep de rank pode ser reportado como
   trabalho futuro.
